# Laboratorio 2 · Bitácora

**Nombre:**  Grupos 5: Jorge Luis, Mónica Velasco, Juan David Cruz 
**Usuario de GitHub:**  monicavelasco01-cloud
**Fecha:**  31/08/2026

---

> Los enunciados están en la guía del laboratorio. Aquí solo van tus
> predicciones, tus resultados y tus explicaciones.

> **La regla:** la predicción se escribe ANTES de ejecutar la celda de código
> que tiene debajo. Equivocarse no resta. Rellenarla después, sí.

> **Lo nuevo de hoy:** los métodos de esta sesión son aleatorios. Una sola
> ejecución no es una medición. A partir del ejercicio 2, todo número que
> escribas aquí tiene que venir con su intervalo y con cuántas semillas lo
> produjeron.


## Preparación


In [1]:
import numpy as np

from rlrs.dp import value_iteration
from rlrs.envs import ARROWS, GridWorld, acantilado
from rlrs.evaluation import evaluate
from rlrs.policies import EpsilonAvidaPolicy, GreedyTabularPolicy
from rlrs.td import error_frente_a, mc_control, q_learning, sarsa

# Si esta celda falla, para y resuélvelo antes de seguir.
print('todo importado')


todo importado


## Mi variante

La misma de ayer. Si no la anotaste, ejecuta `uv run python scripts/variante.py`.


In [2]:
RUIDO = 0.2   # <- rellena, el mismo de ayer
COSTE = -0.1   # <- rellena, el mismo de ayer
GAMMA = 0.9

mi_env = GridWorld(noise=RUIDO, step_reward=COSTE)

# La respuesta conocida: tu V* de ayer. Es contra esto que medimos hoy.
optimos, politica_optima, barridos = value_iteration(mi_env, gamma=GAMMA)
print(f'{barridos} barridos'); print(mi_env.render_values(optimos, politica_optima))


35 barridos
+0.26>  +0.45>  +0.65>  +0.91>   +1    
+0.10^    ###   +0.46^  +0.50^   -1    
-0.02^  +0.09>  +0.26^    ###   -0.28v 
-0.13^  -0.04^  +0.08^  -0.05<  -0.18< 


### Dos ayudas que se usan en todo el cuaderno


In [3]:
libres = [(r, c) for r in range(mi_env.n_rows) for c in range(mi_env.n_cols)
          if not mi_env.is_wall((r, c)) and not mi_env.is_terminal((r, c))]


def coincidencias(q):
    """En cuántas casillas la acción ávida de q es la acción óptima."""
    return sum(int(q[mi_env.state_index(p)].argmax()
                   == politica_optima[mi_env.state_index(p)]) for p in libres)


def intervalo(xs):
    """Media e intervalo de confianza al 95 %. Devuelve (media, bajo, alto)."""
    a = np.array(xs, dtype=float)
    media = a.mean()
    mitad = 1.96 * a.std(ddof=1) / np.sqrt(len(a)) if len(a) > 1 else 0.0
    return media, media - mitad, media + mitad


print(f'{len(libres)} casillas libres')


16 casillas libres


---
## Ejercicio 1 · Los tres métodos contra la respuesta conocida


**Antes de ejecutar.** Ordena los tres métodos de menor a mayor error, y di por qué crees que ese es el orden.

_Tu predicción:_  Según lo que hemos visto, pensaría que este es su orden: 
1. Q-learning 
2. SARSA 
3. Monte Carlo 

lo anterior porque Q-learning es un método fuera de política que utiliza la mejor acción posible para actualizar su estimación, lo que le permite converger más rápidamente y alcanzar un rendimiento óptimo. SARSA, aunque también es eficiente, aprende de la política actual, lo que puede hacer que sea más conservador y menos efectivo en comparación con Q-learning en ciertos entornos. Por otro lado, Monte Carlo, depende de la finalización de episodios completos para aprender, lo que puede resultar en una convergencia más lenta y un mayor error, especialmente en entornos donde no se conocen todos los estados.



In [4]:
for nombre, metodo in (('monte-carlo', mc_control), ('sarsa', sarsa), ('q-learning', q_learning)):
    ap = metodo(mi_env, episodes=5000, gamma=GAMMA, seed=0)
    err = error_frente_a(ap.q, optimos, mi_env)
    print(f'{nombre:<12} error {err:.4f}   política {coincidencias(ap.q)}/{len(libres)}')


monte-carlo  error 0.5158   política 14/16
sarsa        error 0.2489   política 14/16
q-learning   error 0.1959   política 13/16


**Explicación.** ¿Coincidió con tu predicción? Si no, ¿qué esperabas y qué encontraste?

_Tu explicación:_  Los resultados confirman que Q-learning tuvo el menor error, seguido de SARSA, y finalmente Monte Carlo con el mayor error. Esto coincide con la predicción inicial. El método que presenta un mayor error es el de monte-carlo con un valor de 0.5158 con una politica de 14/16. Además se tiene que q-learning tuvo el menor error, seguido de SARSA. El hecho de que Q-learning haya tenido el menor error respalda la afirmación de que este método es más eficiente en este contexto.



---
## Ejercicio 2 · Un número sin intervalo, otra vez

> Esta celda tarda cerca de medio minuto. No se colgó.


**Antes de ejecutar.** 1. ¿Se va a mantener el orden del ejercicio 1 con cinco semillas? 2. ¿Y van las dos cifras, el error y el recuento de política, a contar la misma historia?

_Tu predicción:_  

RTA 1: Es probable que el orden de los métodos se mantenga, con Q-learning en primer lugar, seguido de SARSA y luego Monte Carlo. Sin embargo, al usar múltiples semillas, es posible que los resultados varíen un poco debido a la naturaleza estocástica de los métodos.

RTA 2: Puede que no siempre cuenten con el mismo resultado. El error puede diferir entre métodos, pero el recuento de política podría mostrar un patrón diferente, especialmente si un método tiene un alto error pero aún así selecciona correctamente la acción óptima en muchas casillas.

In [5]:
for nombre, metodo in (('monte-carlo', mc_control), ('sarsa', sarsa), ('q-learning', q_learning)):
    errores, politicas = [], []
    for semilla in range(5):
        ap = metodo(mi_env, episodes=5000, gamma=GAMMA, seed=semilla)
        errores.append(error_frente_a(ap.q, optimos, mi_env))
        politicas.append(coincidencias(ap.q))
    e, elo, ehi = intervalo(errores)
    p, plo, phi = intervalo(politicas)
    print(f'{nombre:<12} error {e:.4f} [{elo:.4f}, {ehi:.4f}]'
          f'   política {p:.1f} [{plo:.1f}, {phi:.1f}]')


monte-carlo  error 0.5238 [0.4918, 0.5558]   política 12.6 [10.8, 14.4]
sarsa        error 0.2359 [0.1807, 0.2912]   política 13.8 [13.1, 14.5]
q-learning   error 0.2169 [0.1969, 0.2369]   política 13.2 [12.5, 13.9]


**Con los intervalos delante, responde las dos por separado.**

1. ¿El **error** distingue a los tres métodos, o hay parejas cuyos intervalos se solapan?

   _Tu respuesta:_ Vemos que el orden de los métodos según el error se mantiene: Q-learning tiene el menor error, seguido de SARSA, y finalmente Monte Carlo con el mayor error. El error sí distingue a los tres métodos. Los intervalos de confianza para Q-learning (0.1969,0.2369) y SARSA (0.1807, 0.2912) no se suporponen, esto nos dice que hay diferencia significativa entre estos dos métodos. Sin embargo, el intervalo de Monte-carlo (0.4918,0.5558) se encuentra or encima de los otros dos, lo que nos dice que es muy distinto y tiene un error mayor. Por lo tanto, los errores de q-learning y SARSA son significativamente menores que el de monte carlo, y todos los métodos son distintos entre sí en términos de error. 

2. ¿El **recuento de política** los distingue?

   _Tu respuesta:_  No, no se distinguen entre los tres métodos. SARSA logró una mayor coincidencia con la política óptima (13.8), seguido de Q-learning (13.2) y finalmente Monte Carlo (12.6). Esto sugiere que, aunque Q-learning tiene el menor error, SARSA es más efectivo en términos de seleccionar la acción óptima en este caso específico, además de que los intervalos de confianza para las políticas también reflejan esto, con SARSA mostrando un rango más estrecho y consistente.



**Explicación.** ¿Coincidió con tu predicción? Si no, ¿qué esperabas y qué encontraste?

_Tu explicación:_  La predicción sobre el orden de los métodos se cumplió, aunque el recuento de política mostró que SARSA puede ser más efectivo en términos de política óptima, a pesar de tener un error ligeramente superior al de Q-learning.



---
## Ejercicio 3 · Apagar la exploración


**Antes de ejecutar.** Con $\varepsilon = 0$ el agente siempre toma la acción que ahora mismo cree mejor. ¿Aprenderá la política óptima, una peor, o depende de la suerte inicial? Y con $\varepsilon = 0{,}5$: ¿mejor o peor que con $0{,}1$?

_Tu predicción:_  Esta predicción depende mucho del valor de $\varepsilon$ que tomemos, por eso, miremos que podría llegar a pasar con cada uno de ellos:

* Con $\varepsilon=0$: El agente siempre tomará la acción que cree que es la mejor según su política actual, lo que significa que no explorará. Esto podría llevar a que el agente se quede atrapado en una política subóptima, ya que no aprenderá de nuevas acciones.

* Con $\varepsilon=0.05$: El agente explorará un 5% del tiempo. Esto debería permitirle aprender más sobre el entorno y potencialmente descubrir mejores acciones. Se espera que el rendimiento sea mejor que con $\varepsilon=0$, pero no tan bueno como si explorara más.

* Con $\varepsilon=0.1$: A medida que $\varepsilon$ aumenta, el agente explorará más. Se espera que el rendimiento mejore con cada incremento de $\varepsilon$, ya que el agente tendrá más oportunidades de aprender de nuevas experiencias.



In [6]:
for eps in (0.0, 0.05, 0.1, 0.3, 0.5):
    errores, politicas = [], []
    for semilla in range(5):
        ap = sarsa(mi_env, episodes=5000, gamma=GAMMA,
                   epsilon=eps, epsilon_final=eps, seed=semilla)   # sin decaimiento
        errores.append(error_frente_a(ap.q, optimos, mi_env))
        politicas.append(coincidencias(ap.q))
    e, elo, ehi = intervalo(errores)
    p, plo, phi = intervalo(politicas)
    print(f'eps {eps:<5} error {e:.4f} [{elo:.4f}, {ehi:.4f}]'
          f'   política {p:.1f} [{plo:.1f}, {phi:.1f}]')


eps 0.0   error 0.1410 [0.0902, 0.1918]   política 13.0 [12.4, 13.6]
eps 0.05  error 0.1972 [0.1491, 0.2452]   política 13.8 [13.1, 14.5]
eps 0.1   error 0.2575 [0.1861, 0.3289]   política 14.4 [13.9, 14.9]
eps 0.3   error 0.3364 [0.2634, 0.4094]   política 14.6 [13.8, 15.4]
eps 0.5   error 0.5259 [0.5112, 0.5406]   política 15.0 [14.4, 15.6]


**Son dos fallos distintos.** Nombra por separado qué le falta al agente de $\varepsilon = 0$ y qué le sobra al de $\varepsilon = 0{,}5$.

_Tu respuesta:_  

**Falta:**  El agente no explora y logra un error de 0.1410, lo que indica que, a pesar de que este no explora tan detallado, tiene un rendimiento relativamente bueno, pero el recuento de politica muestra que no está aprendiendo completamente. A medida que $\varepsilon$ aumenta, el error tiende a aumentar también. Esto es contrario a la expectativa inicial de que un mayor $\varepsilon$ llevaría a un mejor rendimiento. Este fenómeno puede deberse a que el agente comienza a explorar más y, por lo tanto, se expone a acciones que no son óptimas.

**Sobra:** El recuento de política mejora inicialmente con $\varepsilon=0.05$ (13.8) y $\varepsilon=0.1$ (14.4), pero luego se estabiliza en torno a 14.6 y 15.0 con valores más altos de $\varepsilon$. Esto indica que, aunque el agente explora más, no necesariamente mejora su rendimiento en términos de seleccionar la acción óptima.



**Explicación.** ¿Coincidió con tu predicción? Si no, ¿qué esperabas y qué encontraste?

_Tu explicación:_  Las predicciones sobre el comportamiento del agente con diferentes valores de $\varepsilon$ se cumplieron en parte, ya que el rendimiento se vio afectado negativamente con valores altos de $\varepsilon$. Esto resalta la importancia del balance entre exploración y explotación.



---
## Ejercicio 4 · El tamaño del paso


**Antes de ejecutar.** ¿El error va a bajar monótonamente al subir $\alpha$, va a subir monótonamente, o va a tener un mínimo en algún punto intermedio? Apuesta por una de las tres formas.

_Tu predicción:_  Es probable que el error tenga un mínimo en algún punto intermedio. Un valor de $\alpha$ muy pequeño hará que el agente aprenda lentamente, mientras que un valor muy alto puede hacer que el agente olvide rápidamente lo aprendido y cause oscilaciones en el rendimiento.



In [7]:
for alpha in (0.01, 0.1, 0.5, 0.9):
    errores = [error_frente_a(sarsa(mi_env, episodes=5000, gamma=GAMMA,
                                    alpha=alpha, seed=s).q, optimos, mi_env)
               for s in range(5)]
    e, elo, ehi = intervalo(errores)
    print(f'alpha {alpha:<5} error {e:.4f} [{elo:.4f}, {ehi:.4f}]')


alpha 0.01  error 0.2518 [0.2084, 0.2952]
alpha 0.1   error 0.2359 [0.1807, 0.2912]
alpha 0.5   error 0.4566 [0.4122, 0.5009]
alpha 0.9   error 1.1386 [0.9066, 1.3706]


**Distingue los dos problemas.** El de $\alpha$ muy pequeño y el de $\alpha$ muy grande no son el mismo.

_Tu respuesta:_  un $\alpha$ muy pequeño resulta en un aprendizaje lento y conservador, mientras que un $\alpha$ muy grande conduce a un aprendizaje inestable y poco confiable. Ambos problemas son distintos y requieren un ajuste cuidadoso del tamaño del paso para lograr un equilibrio que maximice el rendimiento del agente.



**Explicación.** ¿Coincidió con tu predicción? Si no, ¿qué esperabas y qué encontraste?

_Tu explicación:_  La predicción de que el error tendría un mínimo en algún punto intermedio se cumplió. Los resultados muestran que un  $\alpha$ demasiado bajo o demasiado alto resulta en un mal rendimiento, mientras que un valor moderado (0.1) parece ser el más efectivo.



---
## Ejercicio 5 · El error plantado

Este no lleva código propio. Ejecuta en la terminal:

```
uv run python experiments/sin_modelo.py --parte 3
```


**Antes de ejecutar.** ¿Cuál de las dos formas de medir va a dar un retorno peor, y por qué? ¿Y cuánto peor, un poco o mucho?

_Tu predicción:_  
* **¿Cuál método dará un retorno peor?:** Se espera que SARSA dé un retorno peor en comparación con Q-learning debido a su naturaleza on-policy y su dependencia de las recompensas que recibe.
* **¿Cuánto peor?:** La magnitud de la diferencia dependerá de la severidad del error plantado. Sin embargo, es probable que SARSA experimente un rendimiento mucho peor en comparación con Q-learning, ya que puede quedar atrapado en una política subóptima al seguir aprendiendo de las recompensas erróneas.


Esto significa que, en un entorno con un error plantado significativo, podríamos esperar que SARSA no solo tenga un rendimiento inferior, sino que también sea más ineficaz en su aprendizaje en comparación con Q-learning.



**Pega aquí la salida del guion.**

```
  UPTC · Sesion 2 · Aprender sin modelo del entorno

  3 · El error plantado: medir con la politica que exploraba

  como se mide             retorno                IC 95%   exito
  --------------------------------------------------------------
  avida (epsilon = 0)       +0.654  [+0.640, +0.669]  100.0%
  epsilon = 0.05            +0.646  [+0.631, +0.660]  100.0%
  epsilon = 0.1             +0.617  [+0.596, +0.638]   99.7%
  epsilon = 0.3             +0.429  [+0.380, +0.478]   96.7%

  Es el MISMO agente en las cuatro filas. Lo unico que cambia es si
  sigue explorando mientras se le mide. Con epsilon = 0.3 los
  intervalos ni siquiera se solapan con los de epsilon = 0: la
  conclusion equivocada seria estadisticamente significativa.
```



**El diagnóstico.** ¿Por qué esa medición está mal hecha, y qué está midiendo en realidad? Y en una frase: ¿cuándo sí tendría sentido medir con la política que explora?

_Tu respuesta:_  La medición está mal hecha porque se está evaluando el rendimiento del agente utilizando la política que explora en lugar de la política óptima que el agente debería seguir. Esto puede dar como resultado una evaluación sesgada y poco representativa del verdadero desempeño del agente, ya que se mide el rendimiento en un contexto donde el agente no está siguiendo la mejor acción disponible, lo que puede llevar a conclusiones erróneas sobre la efectividad del aprendizaje.

En realidad, esta medición está capturando el rendimiento del agente en un entorno donde está explorando activamente, lo que puede incluir acciones subóptimas y recompensas erróneas. Por lo tanto, lo que se mide es la capacidad del agente para manejar la exploración y la explotación en presencia de un error plantado, en lugar de su capacidad para seguir la política óptima. Tendría sentido medir con la política que explora cuando se desea evaluar la robustez del agente frente a la incertidumbre y la variabilidad del entorno, o en situaciones donde la exploración es fundamental para el aprendizaje en entornos dinámicos y cambiantes.





---
## Ejercicio 6 · El acantilado


**Antes de ejecutar.** ¿Cuál de los dos métodos va a ganar? Escríbelo, y después vuelve a leer la pregunta: ¿tiene sentido tal como está formulada?

_Tu predicción:_  Lo que espero es que: Q-learning gane en términos de rendimiento, mostrando un menor error y una mayor coincidencia con la política óptima en comparación con SARSA. Esto se debe a que Q-learning, al ser un método off-policy, tiene una mayor capacidad para aprender de acciones exploratorias y adaptarse a entornos dinámicos.



In [8]:
cl = acantilado()

for nombre, metodo in (('sarsa', sarsa), ('q-learning', q_learning)):
    avido, explorando, entrenamiento = [], [], []
    for semilla in range(5):
        ap = metodo(cl, episodes=5000, gamma=1.0, alpha=0.1, seed=semilla)
        ev = evaluate(acantilado(), GreedyTabularPolicy(ap.q.argmax(axis=1)),
                      episodes=100, base_seed=0)
        avido.append(ev.mean)
        ex = evaluate(acantilado(), EpsilonAvidaPolicy(ap.q, 0.05),
                      episodes=100, base_seed=0)
        explorando.append(ex.mean)
        entrenamiento.append(float(np.mean(ap.retornos[-500:])))
    a, alo, ahi = intervalo(avido)
    x, xlo, xhi = intervalo(explorando)
    t, tlo, thi = intervalo(entrenamiento)
    print(f'{nombre:<11} ávido {a:+.2f} [{alo:+.2f}, {ahi:+.2f}]'
          f'   explorando {x:+.2f} [{xlo:+.2f}, {xhi:+.2f}]'
          f'   entrenamiento {t:+.2f} [{tlo:+.2f}, {thi:+.2f}]')


sarsa       ávido -16.00 [-16.00, -16.00]   explorando -16.98 [-17.01, -16.96]   entrenamiento -18.24 [-18.72, -17.75]
q-learning  ávido -12.00 [-12.00, -12.00]   explorando -24.86 [-25.92, -23.80]   entrenamiento -27.01 [-28.37, -25.66]


Para esta parte tenemos que:
1. Rendimiento General: Aunque Q-learning tuvo un mejor rendimiento ávido, su desempeño se deterioró significativamente al explorar y durante el entrenamiento. Esto pone de manifiesto que, aunque Q-learning puede ser más efectivo en ciertas condiciones, su rendimiento puede verse comprometido en entornos con alta variabilidad o error.

2. Estrategias de Aprendizaje: La elección de la estrategia de aprendizaje (exploración vs. explotación) es crucial y puede influir en el rendimiento de manera significativa. En este caso, SARSA parece manejar mejor la exploración en comparación con Q-learning.

### Los dos caminos


In [9]:
for nombre, metodo in (('sarsa', sarsa), ('q-learning', q_learning)):
    ap = metodo(cl, episodes=5000, gamma=1.0, alpha=0.1, seed=0)
    print(f'\n{nombre}:')
    print(cl.render_values(ap.q.max(axis=1), ap.q.argmax(axis=1)))



sarsa:
-14.13>  -12.99>  -11.87>  -10.66>  -9.57>  -8.53>  -7.58>  -6.80>  -5.52>  -4.45>  -3.36>  -2.22v 
-15.37^  -14.40^  -13.58^  -12.47^  -11.95>  -10.16>  -8.90>  -7.71^  -4.53>  -3.41>  -2.44>  -1.02v 
-16.56^  -15.95^  -16.56^  -15.67^  -13.24^  -12.32^  -11.35^  -9.09>  -7.22^  -5.72^  -1.35>  +0.00v 
-17.67^   -100      -100      -100      -100      -100      -100      -100      -100      -100      -100      +0    

q-learning:
-11.60^  -11.03^  -10.32^  -9.52>  -8.66>  -7.79>  -6.87>  -5.92>  -4.96>  -3.98>  -2.99>  -2.00v 
-12.00>  -11.00>  -10.00>  -9.00v  -8.00>  -7.00>  -6.00v  -5.00>  -4.00>  -3.00v  -2.00v  -1.00v 
-11.00>  -10.00>  -9.00>  -8.00>  -7.00>  -6.00>  -5.00>  -4.00>  -3.00>  -2.00>  -1.00>  +0.00v 
-12.00^   -100      -100      -100      -100      -100      -100      -100      -100      -100      -100      +0    


**La explicación del mecanismo.** Escribe las dos reglas de actualización una debajo de la otra y subraya lo único que cambia: qué valor se usa para el estado siguiente. Desde ahí, explica por qué cada método aprende el camino que aprende.

_Tu respuesta:_  Tenemos:

1.  Regla de actualización para SARSA
$Q(s, a) \leftarrow Q(s, a) + \alpha \left( r + \gamma Q(s', a') - Q(s, a) \right)$
Aquí,  $s'$ es el estado siguiente y $a'$ es la acción que se toma en ese estado siguiente según la política actual (que puede ser exploratoria).

2. Regla de actualización para Q-learning
$Q(s, a) \leftarrow Q(s, a) + \alpha \left( r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right)$
En este caso, $s'$ es el estado siguiente, pero se utiliza 
$\max_{a'} Q(s', a')$, es decir, la mejor acción posible desde ese estado siguiente, independientemente de la política actual.

Ahora como se vio en clase, tenemos el porqué toma estos caminos:

**SARSA**

a. Dependencia de la Política: SARSA es un método que actualiza su valor de acción basándose en la política que está siguiendo. Esto implica que el valor de $Q(s', a')$ se calcula utilizando la acción que realmente se toma en el estado siguiente según la política actual.
b. Exploración y Aprendizaje: En el caso de SARSA, si el agente explora (por ejemplo, eligiendo acciones aleatorias), puede terminar aprendiendo caminos que son subóptimos, ya que sigue la política que incluye exploración. Esto es evidente en los resultados que compartiste, donde SARSA muestra un retorno más bajo al explorar, ya que puede estar tomando decisiones que no son las mejores disponibles.

**Q-learning**

a. Maximización de la Recompensa: Q-learning es un método que se basa en la mejor acción posible en el estado siguiente, sin importar la política actual. Esto permite que el agente aprenda de las mejores acciones disponibles en el entorno y no se vea tan afectado por la exploración.
b. Aprendizaje más Eficiente: Debido a esta característica, Q-learning tiende a aprender caminos más óptimos, ya que siempre busca maximizar su recompensa. En los resultados, se puede ver que Q-learning tiene un retorno más alto en comparación con SARSA, especialmente en las etapas iniciales, lo que sugiere que se está acercando más a la política óptima.


**Explicación.** ¿Coincidió con tu predicción? Si no, ¿qué esperabas y qué encontraste?

_Tu explicación:_  Aunque la predicción inicial sobre el rendimiento ávido de Q-learning se confirmó, los resultados en exploración y entrenamiento sorprendieron, ya que se mostro que SARSA puede ser más adecuado en ciertas circunstancias. Esto enfatiza la necesidad de un análisis más profundo al evaluar métodos de aprendizaje por refuerzo en diferentes entornos.



---
## Antes de entregar

- [ ] Las seis predicciones están escritas, y se escribieron antes de ejecutar.
- [ ] Todos los números llevan su intervalo y dicen cuántas semillas los produjeron.
- [ ] Ninguna conclusión dice más de lo que los intervalos permiten decir.
- [ ] Las explicaciones de los ejercicios 5 y 6 hablan del mecanismo, no del resultado.
- [ ] **Kernel → Restart & Run All**, y el cuaderno corre entero de arriba abajo.
- [ ] `git add`, `git commit -m "Laboratorio 2"`, `git push`.
- [ ] Las dos líneas pegadas en Moodle.
